In [0]:
%sql
CREATE TABLE IF NOT EXISTS main.default.pipeline_logs (
    layer_name STRING,
    table_name STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    record_count BIGINT,
    status STRING,
    error_message STRING
)
USING DELTA;

In [0]:
from pyspark.sql.functions import current_timestamp, lit

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, LongType
)

schema = StructType([
    StructField("layer_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("record_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

log_data = [(
    "Silver",
    "sales_transactions",
    start_time,
    end_time,
    record_count,
    "SUCCESS",
    None
)]

spark.createDataFrame(log_data, schema) \
    .write.mode("append") \
    .saveAsTable("main.default.pipeline_logs")

In [0]:
try:
    # processing logic
    pass
except Exception as e:
    spark.createDataFrame(
        [("Silver", "sales_transactions", start_time, current_timestamp(), 0, "FAILED", str(e))],
        ["layer_name", "table_name", "start_time", "end_time", "record_count", "status", "error_message"]
    ).write.mode("append").saveAsTable("main.default.pipeline_logs")
    raise